In [58]:
RR = {}
_debug_ = False

def debug_print(message):
    """Prints debug messages if debugging is enabled."""
    if _debug_:
        print(f"[DEBUG] {message}")

In [8]:
ej = b"127.0.0.1;8881;10;1;0;1024;0;Hello, world!"
a = ej.split(b";", 7)
a

[b'127.0.0.1', b'8881', b'10', b'1', b'0', b'1024', b'0', b'Hello, world!']

In [2]:
a, b = ((1,2),3)
a

(1, 2)

In [75]:
def parse_packet(IP_BRACKETS: bytes) -> dict:
    """
    Extrae los headers y datos del paquete.
    """
    try:
        ip, port, ttl, id, offset, size, flag, msg = IP_BRACKETS.decode().split(";", 7) 
    except ValueError:
        debug_print(f"Error al parsear el paquete: {IP_BRACKETS.decode()}")
        return None
    
    return {
        "ip": ip,
        "port": int(port),
        "ttl": int(ttl),
        "id": int(id),
        "offset": int(offset),
        "size": int(size),
        "flag": int(flag),
        "msg": msg,
    }

def create_packet(structured_data: dict) -> bytes:
    """
    Crea un paquete a partir de la estructura. Si no hay TTL, no lo incluye.
    """
    ip = structured_data["ip"]
    port = structured_data["port"]
    ttl = structured_data['ttl']
    id = structured_data['id']
    offset = structured_data['offset']
    size = structured_data['size']
    flag = structured_data['flag']
    msg = structured_data["msg"]

    packet = f"{ip};{port};{ttl};{id};{offset};{size};{flag};{msg}".encode() # ejemplo: "127.0.0.1;8881;10;1;0;1024;0;Hello, world!"
    return packet

In [11]:
IP_packet_v1 = b"127.0.0.1;8885;10;347;0;5;0;hola!"
parsed_IP_packet = parse_packet(IP_packet_v1)
IP_packet_v2 = create_packet(parsed_IP_packet)
print("IP_packet_v1 == IP_packet_v2 ? {}".format(IP_packet_v1 == IP_packet_v2))

IP_packet_v1 == IP_packet_v2 ? True


In [76]:
def check_routes(routes_files_name: str, destinarion_address: tuple[str, int]) -> tuple[str, int]:
    """
    Función que revisa en orden la tabla de rutas para indicar la adirección del siguiente salto.
    """
    ip, port = destinarion_address
    candidate_routes = []

    with open(routes_files_name, "r") as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith("#"):
                continue  # Ignorar líneas vacías o comentarios
            parts = line.split(" ")  # el formato de la línea es:  RED I_port F_port next_IP next_PORT MTU, por ej 127.0.0.1/24 8881 8882 127.0.0.1 8882 50

            if len(parts) != 6:
                debug_print(f"Línea de ruta inválida: {line}")
                continue  # Ignorar líneas mal formateadas
            dest_ip, initial_port, dest_port, next_hop_ip, next_hop_port, mtu = parts

            if (int(initial_port) <= port <= int(dest_port)): #dest_ip == destinarion_address[0] and ver si comparar la ip o no ya q es solo uno
                debug_print(f"Ruta encontrada para {destinarion_address}: siguiente salto {next_hop_ip}:{next_hop_port}")   
                candidate_routes.append((next_hop_ip, int(next_hop_port)))
    
    if candidate_routes:
        # Seleccionar la ruta con el siguiente salto más cercano (implementación del algoritmo Round Robin)
        key = (initial_port, dest_port)
        i = RR.get(key, 0) % len(candidate_routes)
        RR[key] = (i + 1) % len(candidate_routes) 
        debug_print(f"key: {key}, i: {i}, RR[key]: {RR[key]}, candidate_routes: {candidate_routes}")
        return candidate_routes[i]

    debug_print(f"No se encontró ruta para {destinarion_address}")  
    return None

In [77]:
HEADER_SIZE = 16 

def fragment_IP_packet(IP_packet: bytes, MTU: int) -> list[dict]:
    """
    Función que fragmenta un paquete IP en varios paquetes según el MTU.
    """
    msg_len = MTU - HEADER_SIZE # Tamaño permitido del mensaje en bytes después de restar el tamaño del header
    packet = parse_packet(IP_packet)
    msg = packet["msg"]
    
    if len(msg) <= msg_len:
        debug_print(f"No es necesario fragmentar el paquete: {packet}\n")
        return [parse_packet(IP_packet)]  # No es necesario fragmentar
    # Hay que fragmentar el mensaje
    fragmented_packets = []
    half_size = len(msg) // 2 
    # Resolvemos recursivamente como si fuera un árbol
    left_packet = create_packet({**packet, 'msg': msg[:half_size], 'size': half_size, 'flag': 1}) # Fragmento izquierdo
    right_packet = create_packet({**packet, 'msg': msg[half_size:], 'offset': packet['offset'] + half_size, 'size':len(msg)-half_size}) # Fragmento derecho
    debug_print(f"Fragmentando paquete: {packet}\n en dos fragmentos: {left_packet} y {right_packet}\n")
    
    fragmented_packets += fragment_IP_packet(left_packet, MTU) + fragment_IP_packet(right_packet, MTU) 

    # ------------------------------------------------------------------------------
    # header_size = len(create_packet({**packet, "msg": ""}))  # Tamaño del header sin el mensaje
    # trozo_mensaje_en_bytes = MTU - header_size

    # fragments = []
    # offset = 0
    # while offset < len(msg):
    #     fragment_msg = msg[offset:offset + trozo_mensaje_en_bytes]
    #     fragment_packet = {**packet, "msg": fragment_msg, "offset": offset}
    #     fragments.append(fragment_packet)
    #     offset += trozo_mensaje_en_bytes

    return fragmented_packets

In [51]:
def make_packet(msg_len, id=347):
    msg = "a" * msg_len
    return f"127.0.0.1;8885;10;{id};0;{msg_len};0;{msg}".encode()

casos = [
    (5, 100),    # cabe
    (5, 50),     # cabe
    (122, 150),  # justo sobre el límite
    (122, 100),
    (122, 80),
    (122, 50),
]

for msg_len, mtu in casos:
    packet = make_packet(msg_len)
    fragments = fragment_IP_packet(packet, mtu)
    print(f"mensaje={msg_len} MTU={mtu} total={len(packet)} -> "
          f"{len(fragments)} fragmentos, tamaños: {[len(f) for f in fragments]}")

[DEBUG] No es necesario fragmentar el paquete: {'ip': '127.0.0.1', 'port': 8885, 'ttl': '10', 'id': '347', 'offset': 0, 'size': '5', 'flag': '0', 'msg': 'aaaaa'}

mensaje=5 MTU=100 total=33 -> 1 fragmentos, tamaños: [8]
[DEBUG] No es necesario fragmentar el paquete: {'ip': '127.0.0.1', 'port': 8885, 'ttl': '10', 'id': '347', 'offset': 0, 'size': '5', 'flag': '0', 'msg': 'aaaaa'}

mensaje=5 MTU=50 total=33 -> 1 fragmentos, tamaños: [8]
[DEBUG] No es necesario fragmentar el paquete: {'ip': '127.0.0.1', 'port': 8885, 'ttl': '10', 'id': '347', 'offset': 0, 'size': '122', 'flag': '0', 'msg': 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'}

mensaje=122 MTU=150 total=152 -> 1 fragmentos, tamaños: [8]
[DEBUG] Fragmentando paquete: {'ip': '127.0.0.1', 'port': 8885, 'ttl': '10', 'id': '347', 'offset': 0, 'size': '122', 'flag': '0', 'msg': 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa

In [ ]:
def reassemble_IP_packet(fragment_list: list[dict]) -> bytes:
    """
    Función que reensambla los fragmentos de un paquete IP en un solo paquete.
    """
    if len(fragment_list) == 1:
        if fragment_list[0]['flag'] == fragment_list[0]['offset'] == 0:
            debug_print(f"Paquete sin fragmentar recibido: {fragment_list[0]['msg']}")
            return create_packet(fragment_list[0])
        debug_print(f"Paquete inválido recibido: {fragment_list[0]}")
        return None

    # Ordenar los fragmentos por offset
    fragment_list.sort(key=lambda x: x['offset'])

    if fragment_list[0]['offset'] != 0 or fragment_list[-1]['flag'] != 0:
        debug_print(f"Fragmentos inválidos: primer offset {fragment_list[0]['offset']}, último flag {fragment_list[-1]['flag']}")
        return None

    for anterior, actual in zip(fragment_list, fragment_list[1:]):
        if anterior['offset'] + anterior['size'] != actual['offset']:
            debug_print(f"Falta un fragmento entre offset {anterior['offset']} y {actual['offset']}")
            return None

    # Concatenar los mensajes de los fragmentos
    reassembled_msg = ''.join(
        fragment['msg'] for fragment in fragment_list
    )
    # Crear un nuevo paquete con el mensaje reensamblado y los headers del primer fragmento
    first_fragment = fragment_list[0]
    reassembled_packet = create_packet({
        **first_fragment,
        "offset": 0,
        "size": len(reassembled_msg),
        "flag": 0,
        "msg": reassembled_msg,
    })

    debug_print(f"Paquete reensamblado: {reassembled_packet}")
    return reassembled_packet

In [ ]:
import random

def make_packet(msg: str, id=347):
    return create_packet({"ip": "127.0.0.1", "port": 8885, "ttl": 10, "id": id,
                          "offset": 0, "size": len(msg), "flag": 0, "msg": msg})

# 1. Fragmentar y rearmar, con distintos MTU
for mtu in [100, 80, 55, 50]:
    original = make_packet("a" * 100)
    fragment_list = fragment_IP_packet(original, mtu)
    rearmado = reassemble_IP_packet(fragment_list)
    print(f"MTU={mtu}: {len(fragment_list)} fragmentos, igual al original? {original == rearmado}")
# 2. Fragmentos en desorden
original = make_packet("abcdefghijklmnopqrstuvwxyz" * 4)
fragment_list = fragment_IP_packet(original, 50)
random.shuffle(fragment_list)
print("Desordenado:", original == reassemble_IP_packet(fragment_list))

# 3. Lista incompleta: debe dar None
fragment_list = fragment_IP_packet(original, 50)
print("Sin el último fragmento:", reassemble_IP_packet(fragment_list[:-1]))
print("Sin el primero:", reassemble_IP_packet(fragment_list[1:]))
print("Solo uno:", reassemble_IP_packet(fragment_list[:1]))

# 4. Paquete sin fragmentar (lista de 1 elemento completa)
chico = make_packet("hola!")
print("Sin fragmentar:", chico == reassemble_IP_packet(fragment_IP_packet(chico, 100)))

# 5. Dos IDs distintos, cada lista por separado
p1 = make_packet("a" * 90, id=1)
p2 = make_packet("b" * 90, id=2)
l1 = fragment_IP_packet(p1, 50)
l2 = fragment_IP_packet(p2, 50)
print("ID 1:", p1 == reassemble_IP_packet(l1))
print("ID 2:", p2 == reassemble_IP_packet(l2))

0 50 1 50 aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa
50 50 0 50 aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa
MTU=100: 2 fragmentos, igual al original? True
0 50 1 50 aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa
50 50 0 50 aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa
MTU=80: 2 fragmentos, igual al original? True
0 25 1 25 aaaaaaaaaaaaaaaaaaaaaaaaa
25 25 1 25 aaaaaaaaaaaaaaaaaaaaaaaaa
50 25 1 25 aaaaaaaaaaaaaaaaaaaaaaaaa
75 25 0 25 aaaaaaaaaaaaaaaaaaaaaaaaa
MTU=55: 4 fragmentos, igual al original? True
0 25 1 25 aaaaaaaaaaaaaaaaaaaaaaaaa
25 25 1 25 aaaaaaaaaaaaaaaaaaaaaaaaa
50 25 1 25 aaaaaaaaaaaaaaaaaaaaaaaaa
75 25 0 25 aaaaaaaaaaaaaaaaaaaaaaaaa
MTU=50: 4 fragmentos, igual al original? True
0 26 1 26 abcdefghijklmnopqrstuvwxyz
26 26 1 26 abcdefghijklmnopqrstuvwxyz
52 26 1 26 abcdefghijklmnopqrstuvwxyz
78 26 0 26 abcdefghijklmnopqrstuvwxyz
Desordenado: True
0 26 1 26 abcdefghijklmnopqrstuvwxyz
26 26 1 26 abcdefghijklmnopqrstuvwxyz
52 26 1 26 abcdefghijklmnopqrst

: 